# Preprocessing cleaned
Cleaned version of *aleimba_vectyfi_radar_eda.ipynb*, but also some shortcuts.
Includes OHE, still some #TODO in here.

## Imports

In [66]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.feature_selection import SelectPercentile, mutual_info_regression
from sklearn.compose import make_column_transformer, make_column_selector
from sklearn.pipeline import make_pipeline

from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

from xgboost import XGBClassifier

In [41]:
RANDOM_STATE = 42

## Raw data loading

In [2]:
# Load all data to create smaller sample data sets
df_raw_all = pd.read_csv('../raw_data/export_CAN_2023_2018.csv')

/tmp/ipykernel_131619/414305899.py:2: DtypeWarning: Columns (0: CAE_GPA_ANNEX, 1: ISO_COUNTRY_CODE_GPA, 2: ISO_COUNTRY_CODE_ALL, 3: EU_INST_CODE) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw_all = pd.read_csv('../raw_data/export_CAN_2023_2018.csv')


In [3]:
df_raw_all.shape

(6198063, 75)

# Downsample balanced data set
Balanced between awarded and non-awarded (and for the non-awarded also balanced between 'PROCUREMENT_UNSUCCESSFUL' and 'PROCUREMENT_DISCONTINUED')

In [8]:
values = {'INFO_ON_NON_AWARD': 'awarded'}
df_raw_all = df_raw_all.fillna(value=values)

In [9]:
df_raw_all[['INFO_ON_NON_AWARD']].value_counts(dropna=False)

INFO_ON_NON_AWARD       
awarded                     5212924
PROCUREMENT_UNSUCCESSFUL     794911
PROCUREMENT_DISCONTINUED     190228
Name: count, dtype: int64

In [10]:
# create max data set, many rows will be removed in duplicate removal below anyways!
n_awarded      = 380_000
n_unsuccessful = 190_000
n_discontinued = 190_000

In [11]:
grp_awarded      = df_raw_all[df_raw_all["INFO_ON_NON_AWARD"] == "awarded"]
grp_unsuccessful = df_raw_all[df_raw_all["INFO_ON_NON_AWARD"] == "PROCUREMENT_UNSUCCESSFUL"]
grp_discontinued = df_raw_all[df_raw_all["INFO_ON_NON_AWARD"] == "PROCUREMENT_DISCONTINUED"]

In [12]:
df_raw_all_balanced = (
    pd.concat([
        grp_awarded.sample(n=n_awarded, random_state=42),
        grp_unsuccessful.sample(n=n_unsuccessful, random_state=42),
        grp_discontinued.sample(n=n_discontinued, random_state=42),
    ])
    # .sample(frac=1, random_state=42) # optional shuffle
    .reset_index(drop=True)
)

In [13]:
df_raw_all_balanced[['INFO_ON_NON_AWARD']].value_counts(dropna=False)

INFO_ON_NON_AWARD       
awarded                     380000
PROCUREMENT_UNSUCCESSFUL    190000
PROCUREMENT_DISCONTINUED    190000
Name: count, dtype: int64

In [14]:
df_raw_all_balanced.shape

(760000, 75)

In [15]:
df_raw_all_balanced.to_csv('../raw_data/export_CAN_2023_2018_balanced_760k.tsv', sep='\t', index=False)

## Preprocessing with downsampled balanced 760k data set

In [2]:
# df_raw_all_balanced reload balanced 760k data set, to work from here
df = pd.read_csv('../raw_data/export_CAN_2023_2018_balanced_760k.tsv',
                 sep='\t',
                 low_memory=False)

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 760000 entries, 0 to 759999
Data columns (total 75 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   ID_NOTICE_CAN                 760000 non-null  int64  
 1   TED_NOTICE_URL                760000 non-null  str    
 2   YEAR                          760000 non-null  int64  
 3   ID_TYPE                       760000 non-null  int64  
 4   DT_DISPATCH                   760000 non-null  str    
 5   XSD_VERSION                   760000 non-null  str    
 6   CANCELLED                     760000 non-null  int64  
 7   CORRECTIONS                   760000 non-null  int64  
 8   B_MULTIPLE_CAE                751922 non-null  str    
 9   CAE_NAME                      760000 non-null  str    
 10  CAE_NATIONALID                504611 non-null  str    
 11  CAE_ADDRESS                   748576 non-null  str    
 12  CAE_TOWN                      759999 non-null  str    


### Get rid of tender-awarded columns
'contract award' columns (from PDF *TED_csv_data_information_v3.6.1.pdf*) plus additional ones that most likely have data leakage

In [4]:
# keep target col 'INFO_ON_NON_AWARD'

# 'GPA_COVERAGE' contains also data leakage -> category 6 "6: not covered by GPA because contract was Non awarded (All lots of CAN are unsuccessful or discontinued)"
# `NUMBER_AWARDS` notice - count of `ID_AWARD` rows for this CAN -> **Not known at bid time** — only knowable after awards are published
# final contract values 'VALUE_EURO', 'VALUE_EURO_FIN_1', 'VALUE_EURO_FIN_2'
# get rid of 'XSD_VERSION' has nothing to do with tenders (just the version of the CSV file format)
# get rid of 'DT_DISPATCH' as datetime64 type and overlaps with YEAR
tender_awarded_cols = ['GPA_COVERAGE', 'NUMBER_AWARDS', 'VALUE_EURO', 'VALUE_EURO_FIN_1', 'VALUE_EURO_FIN_2', 'XSD_VERSION', 'DT_DISPATCH',
                       'ID_AWARD', 'ID_LOT_AWARDED', 'INFO_UNPUBLISHED', 'B_AWARDED_TO_A_GROUP', 'WIN_NAME', 'WIN_NATIONALID', 'WIN_ADDRESS', 'WIN_TOWN', 'WIN_POSTAL_CODE', 'WIN_COUNTRY_CODE', 'B_CONTRACTOR_SME', 'CONTRACT_NUMBER', 'TITLE', 'NUMBER_OFFERS', 'NUMBER_TENDERS_SME', 'NUMBER_TENDERS_OTHER_EU', 'NUMBER_TENDERS_NON_EU', 'NUMBER_OFFERS_ELECTR', 'AWARD_EST_VALUE_EURO', 'AWARD_VALUE_EURO', 'AWARD_VALUE_EURO_FIN_1', 'B_SUBCONTRACTED', 'DT_AWARD']
len(tender_awarded_cols)

30

In [5]:
df.drop(columns=tender_awarded_cols, inplace=True)
df.shape

(760000, 45)

### Remove duplicate rows
Remove duplicate rows with the same 'ID_NOTICE_CAN'

In [32]:
# TODO keep number of rows as feature?
# df.ID_NOTICE_CAN_count = df.groupby(['ID_NOTICE_CAN'])['ID_NOTICE_CAN'].count()

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 760000 entries, 0 to 759999
Data columns (total 45 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   ID_NOTICE_CAN                 760000 non-null  int64  
 1   TED_NOTICE_URL                760000 non-null  str    
 2   YEAR                          760000 non-null  int64  
 3   ID_TYPE                       760000 non-null  int64  
 4   CANCELLED                     760000 non-null  int64  
 5   CORRECTIONS                   760000 non-null  int64  
 6   B_MULTIPLE_CAE                751922 non-null  str    
 7   CAE_NAME                      760000 non-null  str    
 8   CAE_NATIONALID                504611 non-null  str    
 9   CAE_ADDRESS                   748576 non-null  str    
 10  CAE_TOWN                      759999 non-null  str    
 11  CAE_POSTAL_CODE               747215 non-null  str    
 12  CAE_GPA_ANNEX                 621379 non-null  str    


In [7]:
df.drop_duplicates(subset=['ID_NOTICE_CAN'], keep='first', inplace=True)
df.shape

(378443, 45)

In [8]:
# now drop the ['ID_NOTICE_CAN'] ID column, not needed anymore
df.drop(columns=['ID_NOTICE_CAN'], inplace=True)

### Missing values

In [9]:
df.isnull().sum().sort_values(ascending=False)

ISO_COUNTRY_CODE_ALL            378272
EU_INST_CODE                    377194
B_ACCELERATED                   368783
FRA_ESTIMATED                   314866
CRIT_PRICE_WEIGHT               253426
CRIT_WEIGHTS                    207333
CRIT_CRITERIA                   206252
ID_LOT                          169289
CAE_NATIONALID                  147067
ADDITIONAL_CPVS                  78991
MAIN_CPV_CODE_GPA                68349
CAE_GPA_ANNEX                    68349
ISO_COUNTRY_CODE_GPA             68349
CRIT_CODE                        28440
B_DYN_PURCH_SYST                 22627
B_GPA                            19901
TAL_LOCATION_NUTS                14123
B_EU_FUNDS                       12930
B_ELECTRONIC_AUCTION              9935
CAE_ADDRESS                       7294
CAE_POSTAL_CODE                   6840
B_INVOLVES_JOINT_PROCUREMENT      6302
B_AWARDED_BY_CENTRAL_BODY         6302
B_MULTIPLE_COUNTRY                5115
B_MULTIPLE_CAE                    5115
LOTS_NUMBER              

In [10]:
# remove columns with many missing values
# 'CAE_NATIONALID' most likely overlaps with 'ISO_COUNTRY_CODE' -> remove
# 'ID_LOT' has integers and strings, not sure what that column does -> remove
# !! 'CRIT_CRITERIA' info on award criteria, however long strings, thus remove for now
# !! 'CRIT_WEIGHTS' remove also because 'CRIT_CRITERIA' removed
missing_value_cols = ['CAE_NATIONALID', 'ISO_COUNTRY_CODE_ALL', 'EU_INST_CODE', 'FRA_ESTIMATED', 'ID_LOT', 'CRIT_CRITERIA', 'CRIT_WEIGHTS']
df.drop(columns=missing_value_cols, inplace=True)

In [11]:
# for 'CRIT_PRICE_WEIGHT' replace str values and null values with '0'
print(df.CRIT_PRICE_WEIGHT.dropna().unique())
df.CRIT_PRICE_WEIGHT = pd.to_numeric(df.CRIT_PRICE_WEIGHT, errors='coerce', downcast='integer').fillna(0)

<StringArray>
[                         '60',                          '65',
                          '30',                          '50',
             'V. DISCIPLINARE',                          '70',
                          '40',                        '70 %',
                        '60.0',                         '100',
 ...
     'voir guide de sélection',        'naj. ponudb. cena 70',
                      '99,00%',                         'Y %',
                        '26,5',                 '40 % - 60 %',
                'Angives ikke',           'Mandatory Pricing',
 'U ΣΒ = U ΤΠ *85% + ΒΟΠ *15%',                    '5 Punkte']
Length: 2493, dtype: str


In [12]:
# actually a binary feature, should also work with OHE (as the ones below)
# 'B_ACCELERATED' a binary feature with many null values, thus in prep for below replace null values with 0
df.B_ACCELERATED = df.B_ACCELERATED.fillna(0)

In [13]:
# features still including null values
feat_non_null = df.isnull().sum()
feat_non_null = list(feat_non_null[feat_non_null > 0].index)

### Binary features

In [14]:
feat_categorical_nunique = df.select_dtypes(include=('str', 'object')).nunique()
feat_categorical_bool = list(feat_categorical_nunique[feat_categorical_nunique == 2].index)
feat_categorical_bool

['B_MULTIPLE_CAE',
 'B_MULTIPLE_COUNTRY',
 'B_ON_BEHALF',
 'B_INVOLVES_JOINT_PROCUREMENT',
 'B_AWARDED_BY_CENTRAL_BODY',
 'B_FRA_AGREEMENT',
 'B_FRA_CONTRACT',
 'B_DYN_PURCH_SYST',
 'B_GPA',
 'B_EU_FUNDS',
 'B_ACCELERATED',
 'CRIT_CODE',
 'B_ELECTRONIC_AUCTION']

### Remove categorical features > 10 categories

In [24]:
feat_categorical_nunique = df.select_dtypes(include=('str', 'object')).nunique()
feat_categorical_large = list(feat_categorical_nunique[feat_categorical_nunique > 9].index)

In [25]:
# remove, except for 'ISO_COUNTRY_CODE' and 'MAIN_ACTIVITY', important features
# 'CAE_TYPE' differently handled below
feat_categorical_large.remove('ISO_COUNTRY_CODE')
feat_categorical_large.remove('MAIN_ACTIVITY')
feat_categorical_large.remove('CAE_TYPE')
feat_categorical_large

['TED_NOTICE_URL',
 'CAE_NAME',
 'CAE_ADDRESS',
 'CAE_TOWN',
 'CAE_POSTAL_CODE',
 'ISO_COUNTRY_CODE_GPA',
 'TAL_LOCATION_NUTS',
 'ADDITIONAL_CPVS']

In [27]:
df.drop(columns=feat_categorical_large, inplace=True)

## sklearn pre-processing pipeline

In [69]:
df.B_ACCELERATED = df.B_ACCELERATED.astype(str)

In [70]:
df.info()

<class 'pandas.DataFrame'>
Index: 378443 entries, 0 to 759996
Data columns (total 29 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   YEAR                          378443 non-null  int64  
 1   ID_TYPE                       378443 non-null  int64  
 2   CANCELLED                     378443 non-null  int64  
 3   CORRECTIONS                   378443 non-null  int64  
 4   B_MULTIPLE_CAE                373328 non-null  str    
 5   CAE_GPA_ANNEX                 310094 non-null  str    
 6   ISO_COUNTRY_CODE              378443 non-null  str    
 7   B_MULTIPLE_COUNTRY            373328 non-null  str    
 8   CAE_TYPE                      378443 non-null  str    
 9   MAIN_ACTIVITY                 378439 non-null  str    
 10  B_ON_BEHALF                   377738 non-null  str    
 11  B_INVOLVES_JOINT_PROCUREMENT  372141 non-null  str    
 12  B_AWARDED_BY_CENTRAL_BODY     372141 non-null  str    
 13  

In [71]:
X = df.drop(columns=['INFO_ON_NON_AWARD'])
y = df.INFO_ON_NON_AWARD
X.shape, y.shape

((378443, 28), (378443,))

In [72]:
# convert target to binary '1' for 'awarded' and '0' for non-awarded ('PROCUREMENT_DISCONTINUED' and 'PROCUREMENT_UNSUCCESSFUL')
y = (y == 'awarded').astype(int)
y.value_counts(normalize=True, dropna=False)

INFO_ON_NON_AWARD
1    0.621697
0    0.378303
Name: proportion, dtype: float64

In [74]:
# remaining categorical columns
feat_categorical_nunique = X.select_dtypes(include=('str', 'object')).nunique()
print(feat_categorical_nunique)
feat_categorical_small = list(feat_categorical_nunique.index)

B_MULTIPLE_CAE                   2
CAE_GPA_ANNEX                    4
ISO_COUNTRY_CODE                33
B_MULTIPLE_COUNTRY               2
CAE_TYPE                        10
MAIN_ACTIVITY                   68
B_ON_BEHALF                      2
B_INVOLVES_JOINT_PROCUREMENT     2
B_AWARDED_BY_CENTRAL_BODY        2
TYPE_OF_CONTRACT                 3
B_FRA_AGREEMENT                  2
B_FRA_CONTRACT                   2
B_DYN_PURCH_SYST                 2
B_GPA                            2
B_EU_FUNDS                       2
TOP_TYPE                         9
B_ACCELERATED                    2
CRIT_CODE                        2
B_ELECTRONIC_AUCTION             2
dtype: int64


In [75]:
preproc_numerical = make_pipeline(
    # KNNImputer(), # runs too long
    SimpleImputer(strategy="mean"),
    MinMaxScaler()
)

preproc_cat = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore", drop="if_binary", sparse_output=False)
)

preproc_transformer = make_column_transformer(
    (preproc_numerical, make_column_selector(dtype_include=["number"])),
    (preproc_cat, make_column_selector(dtype_include=["str", "object"])),
    remainder="drop"
)

preproc_transformer

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('pipeline-1', ...), ('pipeline-2', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_

In [80]:
X_preproc = preproc_transformer.fit_transform(X)

In [78]:
preproc_selector = SelectPercentile(
    mutual_info_regression,
    percentile=80, # keep 80% of all features
)

preproc = make_pipeline(
    preproc_transformer,
    preproc_selector
).set_output(transform="pandas")

preproc

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('columntransformer', ...), ('selectpercentile', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('pipeline-1', ...), ('pipeline-2', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of

In [ ]:
# like KNNImputer runs too long right now ...
X_preproc = preproc.fit_transform(X, y)

KeyboardInterrupt: 

In [81]:
X_train, X_test, y_train, y_test = train_test_split(X_preproc, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

In [82]:
xgb_model = XGBClassifier(
    n_estimators=100, max_depth=9,
    tree_method='hist', enable_categorical=True,
    random_state=42, n_jobs=-1,
)
xgb_model.fit(X_train, y_train)

xgb_roc_auc = roc_auc_score(y_test, xgb_model.predict_proba(X_test)[:, 1])
xgb_accuracy = accuracy_score(y_test, xgb_model.predict(X_test))

print(f'XGB ROC AUC:      {xgb_roc_auc:.4f}')
print(f'XGB Accuracy: {xgb_accuracy*100:.1f}%')

XGB ROC AUC:      0.7688
XGB Accuracy: 71.3%


In [ ]:
# score_baseline = cross_val_score(pipe_baseline, X, y, cv=5, scoring=rmsle).mean()
# score_baseline